In [1]:
import os
import yaml
import pandas as pd
import numpy as np
import glob
import gzip
import json
import gc
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ============================================================
# [목적]
#   5분 단위 raw 데이터를 machine_id + hour 기준으로 집계
#   -> duration이 수천~수만 초로 다양해짐
#   -> hour 피처가 핵심 피처로 부상
#   -> RF 외삽 문제 자연스럽게 해결
#
# [저장 파일]
#   instance_usage_hourly_processed.parquet (기존 파일과 별개)
# ============================================================

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT      = f"/content/drive/MyDrive/{config['project_name']}"
RAW_DATA_PATH   = os.path.join(DRIVE_ROOT, config['paths']['raw_data'])
PROCESSED_PATH  = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])

# 저장 파일명 (기존 parquet과 별개)
OUTPUT_FILE = os.path.join(PROCESSED_PATH, "instance_usage_hourly_processed.parquet")

In [5]:
def process_and_aggregate(file_path: str) -> pd.DataFrame:
    """
    raw json.gz 파일 1개를 로드해서
    machine_id + hour_bucket 기준으로 즉시 집계 후 반환

    Args:
        file_path: raw 파일 경로

    Returns:
        집계된 DataFrame
    """
    # 1. 로드
    data = []
    with gzip.open(file_path, 'rt') as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)

    # 2. 시간 변환 (마이크로초 -> 초)
    df['start_time_sec'] = df['start_time'].astype(float) / 1_000_000
    df['end_time_sec']   = df['end_time'].astype(float)   / 1_000_000

    # 3. CPU / Memory 추출
    df['cpu']    = df['average_usage'].apply(
        lambda x: x.get('cpus', 0) if isinstance(x, dict) else 0
    )
    df['memory'] = df['average_usage'].apply(
        lambda x: x.get('memory', 0) if isinstance(x, dict) else 0
    )

    # 4. hour 추출 (0~23시)
    df['hour']        = (df['start_time_sec'] // 3600).astype(int) % 24
    df['hour_bucket'] = (df['start_time_sec'] // 3600).astype(int)

    # 5. duration 계산
    df['duration'] = (df['end_time_sec'] - df['start_time_sec']).clip(lower=0)

    # 6. power_w / energy_kwh 계산
    df['power_w']    = 200 + (df['cpu'] * 300) + (df['memory'] * 50)
    df['energy_kwh'] = df['power_w'] * (df['duration'] / 3600) / 1000

    # 7. 이상치 제거
    df = df[(df['duration'] > 0) & (df['cpu'] >= 0) & (df['memory'] >= 0)]

    # 8. machine_id + hour_bucket 기준 집계 (즉시!)
    df_agg = df.groupby(['machine_id', 'hour_bucket']).agg(
        cpu        = ('cpu',        'mean'),
        memory     = ('memory',     'mean'),
        hour       = ('hour',       'first'),
        duration   = ('duration',   'sum'),
        energy_kwh = ('energy_kwh', 'sum'),
        power_w    = ('power_w',    'mean'),
    ).reset_index()

    return df_agg

In [6]:
# ============================================================
# 파일별 처리 + 즉시 집계 (메모리 크래쉬 방지)
# ============================================================
all_files = sorted(glob.glob(os.path.join(RAW_DATA_PATH, "instance_usage-*.json.gz")))
print(f"Total files: {len(all_files)}")

hourly_chunks = []

for file_path in tqdm(all_files, desc="Processing"):
    df_agg = process_and_aggregate(file_path)
    hourly_chunks.append(df_agg)
    print(f"  {os.path.basename(file_path)}: {len(df_agg):,} rows after aggregation")
    gc.collect()   # 즉시 메모리 해제

# ============================================================
# 전체 합치기 + 재집계 (파일 경계 처리)
# machine_id + hour_bucket 이 같으면 다시 합산
# ============================================================
print("\nMerging all chunks...")
df_all = pd.concat(hourly_chunks, ignore_index=True)
del hourly_chunks; gc.collect()

print(f"Before final aggregation: {len(df_all):,}")

df_hourly = df_all.groupby(['machine_id', 'hour_bucket']).agg(
    cpu        = ('cpu',        'mean'),
    memory     = ('memory',     'mean'),
    hour       = ('hour',       'first'),
    duration   = ('duration',   'sum'),
    energy_kwh = ('energy_kwh', 'sum'),
    power_w    = ('power_w',    'mean'),
).reset_index()

del df_all; gc.collect()

print(f"After final aggregation: {len(df_hourly):,}")


Total files: 5


Processing:  20%|██        | 1/5 [02:26<09:44, 146.13s/it]

  instance_usage-000000000000.json.gz: 425,886 rows after aggregation


Processing:  40%|████      | 2/5 [04:52<07:19, 146.37s/it]

  instance_usage-000000000001.json.gz: 258,153 rows after aggregation


...

Processing:  80%|████████  | 4/5 [09:24<02:19, 139.55s/it]

  instance_usage-000000000003.json.gz: 308,099 rows after aggregation


Processing: 100%|██████████| 5/5 [11:52<00:00, 142.52s/it]

  instance_usage-000000000004.json.gz: 289,376 rows after aggregation

Merging all chunks...
Before final aggregation: 1,593,087


After final aggregation: 1,454,606


In [7]:
# ============================================================
# 통계 확인
# ============================================================
print(f"\n[duration 통계]")
print(df_hourly['duration'].describe())
print(f"\n[hour 분포]")
print(df_hourly['hour'].value_counts().sort_index())
print(f"\n[energy_kwh 통계]")
print(df_hourly['energy_kwh'].describe())


[duration 통계]
count    1.454606e+06
mean     3.265929e+03
std      1.804283e+03
min      1.000000e+00
25%      3.065000e+03
50%      3.600000e+03
75%      3.600000e+03
max      2.353400e+04
Name: duration, dtype: float64

[hour 분포]
hour
0     77056
1     69774
2     68464
3     65663
4     64746
5     70573
6     66568
7     62277
8     52244
9     52313
10    52241
11    51396
12    51028
13    50895
14    52364
15    52858
16    52960
17    53432
18    55130
19    61097
20    62925
21    70556
22    67871
23    70175
Name: count, dtype: int64

[energy_kwh 통계]
count    1.454606e+06
mean     1.836814e-01
std      1.012198e-01
min      5.555556e-05
25%      1.730811e-01
50%      2.015594e-01
75%      2.029016e-01
max      1.322497e+00
Name: energy_kwh, dtype: float64


In [8]:
# ============================================================
# 저장
# ============================================================
df_hourly.to_parquet(OUTPUT_FILE, index=False)
print(f"\nSaved: {OUTPUT_FILE}")
print(f"Size : {os.path.getsize(OUTPUT_FILE) / (1024**2):.1f} MB")
print("Done!")


Saved: /content/drive/MyDrive/EcoTracing/data/processed/instance_usage_hourly_processed.parquet
Size : 33.8 MB
Done!
